<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/LoRA_%2B_SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

In [ ]:
!pip install trl[peft]

In [ ]:
!pip install --upgrade torchao

In [ ]:
from datasets import load_dataset
from peft import PeftModel, PeftConfig, LoraConfig
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
train_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:200]") # Using the train dataset.
eval_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:200]") # Using the test split for evaluation
model_name = "HuggingFaceTB/SmolLM2-135M" # The model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define a chat template for the tokenizer to process the "messages" field
tokenizer.chat_template = "{% for message in messages %}{{ message['role'].capitalize() }}: {{ message['content'] }}\n{% endfor %}{{ eos_token }}"

In [ ]:
# print(dataset.features)

In [ ]:
# def format_instruction_dataset(example):
#     # This function formats the 'messages' field from the dataset
#     # into a single string suitable for instruction fine-tuning.
#     formatted_text = ""
#     for message in example['messages']:
#         role = message['role']
#         content = message['content']
#         # Simple template, can be adjusted based on model's expected format
#         formatted_text += f"{role.capitalize()}: {content}\n"
#     return formatted_text + tokenizer.eos_token

In [ ]:
peft_config = LoraConfig(
    r=6,
    lora_dropout=0.05, # Dropout probability for LoRA layers
    lora_alpha=12, # usually 2x the rank value
    bias="none", # Recommened and Default, might want to look more into this
    task_type="CAUSAL_LM",
    target_modules="all-linear", # Which modules to apply LoRA to
)

training_args = SFTConfig(
    output_dir="./sft-lora-test",
    learning_rate = 2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=5,
    max_length=512,
    do_eval=True, # Enable evaluation during training
    eval_strategy ="epoch", # Evaluate at the end of each epoch
)

# Load the model
model = AutoModelForCausalLM.from_pretrained(model_name)

trainer = SFTTrainer(
    model = model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, # Pass the evaluation dataset
    peft_config=peft_config,
    args=training_args,
    # max_seq_length=max_seq_length # Maximum sequence length
    processing_class=tokenizer, # Tokenizer will now handle chat templating via its chat_template attribute
    # Removed formatting_func as tokenizer.chat_template handles it
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.826884,1.834471,2.063008,102362.000000,0.574597
2,1.911153,1.829805,2.045502,204724.000000,0.575742
3,1.771698,1.827967,2.037946,307086.000000,0.575752


TrainOutput(global_step=75, training_loss=1.8572466150919595, metrics={'train_runtime': 283.6314, 'train_samples_per_second': 2.115, 'train_steps_per_second': 0.264, 'total_flos': 199130362675200.0, 'train_loss': 1.8572466150919595, 'epoch': 3.0})

In [ ]:
print('Evaluation results:')
trainer.evaluate()

Evaluation results:


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
1.771698,1.827967,3,2.037946,307086.000000,0.575752


{'eval_loss': 1.8279671669006348,
 'eval_entropy': 2.0379456424713136,
 'eval_num_tokens': 307086.0,
 'eval_mean_token_accuracy': 0.5757521033287049}

In [ ]:
# Save the fine-tuned adapter
output_dir = "./sft-lora-test/adapter"
trainer.model.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

LoRA adapter saved to ./sft-lora-test/merged_model


In [ ]:
# from peft import PeftModel, PeftConfig

# # Load a fresh base model
# new_base_model = AutoModelForCausalLM.from_pretrained(model_name)

# # Load the PeftModel (adapter) from the saved directory
# reloaded_model = PeftModel.from_pretrained(new_base_model, output_dir)

# print("LoRA adapter reloaded successfully :)")
# print(reloaded_model)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LoRA adapter reloaded successfully!
PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 576)
        (layers): ModuleList(
          (0-29): 30 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=576, out_features=576, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=576, out_features=6, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=6, out_features=576, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       